# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Musadiq8699/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import duckdb
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

# ==============================================================================
# STEP 1: FETCH DATA FROM HUGGING FACE WAREHOUSE VIA DUCKDB
# ==============================================================================

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

api = HfApi()
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)

fact_files = [f for f in repo_files if f.startswith("fact_content_daily_performance/") and f.endswith(".parquet")]
print(f"Found {len(fact_files)} partition file(s) for the fact table.")

local_paths = [
    hf_hub_download(
        repo_id=REPO_ID,
        filename=f,
        repo_type="dataset",
        token=HF_TOKEN
    )
    for f in fact_files
]

con = duckdb.connect()
con.execute(f"CREATE VIEW fact_table AS SELECT * FROM read_parquet({local_paths})")


# ==============================================================================
# STEP 2: AGGREGATE METRICS FOR DECOMPOSING PAGES LANE USING ACTUAL COLUMNS
# ==============================================================================

query = """
WITH max_date_cte AS (
    SELECT MAX(report_date) AS max_date FROM fact_table
),
page_aggregates AS (
    SELECT
        content_hash_id,

        -- Recent 30-day window metrics (relative to max date in dataset)
        SUM(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                 THEN gsc_clicks ELSE 0 END) AS clicks_last_30d,

        SUM(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                 THEN gsc_impressions ELSE 0 END) AS impressions_last_30d,

        AVG(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                 THEN gsc_avg_position ELSE NULL END) AS position_last_30d,

        SUM(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                 THEN ga4_pageviews ELSE 0 END) AS pageviews_last_30d,

        -- Previous 30-day window metrics (31 to 60 days ago)
        SUM(CASE WHEN report_date < (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                  AND report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '60 days'
                 THEN gsc_clicks ELSE 0 END) AS clicks_prev_30d,

        SUM(CASE WHEN report_date < (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                  AND report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '60 days'
                 THEN gsc_impressions ELSE 0 END) AS impressions_prev_30d,

        COUNT(DISTINCT report_date) AS active_days_count
    FROM fact_table
    GROUP BY content_hash_id
)
SELECT * FROM page_aggregates;
"""

df_raw = con.execute(query).df()
print(f"Raw page-level aggregated data shape: {df_raw.shape}")


# ==============================================================================
# STEP 3: ENGINEER FEATURE VECTOR (DECOMPOSITION SIGNALS)
# ==============================================================================

df = df_raw.copy()

# 1. Decay Signals (Click & Impression drops)
df['click_decay_ratio'] = df['clicks_last_30d'] / (df['clicks_prev_30d'] + 1e-5)
df['impression_decay_ratio'] = df['impressions_last_30d'] / (df['impressions_prev_30d'] + 1e-5)

# 2. CTR & Performance Signals
df['ctr_last_30d'] = df['clicks_last_30d'] / (df['impressions_last_30d'] + 1e-5)
df['expected_ctr'] = 1 / (df['position_last_30d'] + 1e-5)
df['ctr_gap'] = df['ctr_last_30d'] - df['expected_ctr']

# 3. Handle infinities & nulls
df['pageviews_last_30d'] = df['pageviews_last_30d'].fillna(0)
df['click_decay_ratio'] = df['click_decay_ratio'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
df['impression_decay_ratio'] = df['impression_decay_ratio'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
df['position_last_30d'] = df['position_last_30d'].fillna(df['position_last_30d'].median())
df['ctr_gap'] = df['ctr_gap'].fillna(0.0)

# 4. Construct Final Feature Vector (X)
feature_cols = [
    'clicks_last_30d',
    'impressions_last_30d',
    'pageviews_last_30d',
    'position_last_30d',
    'click_decay_ratio',
    'impression_decay_ratio',
    'ctr_last_30d',
    'ctr_gap',
    'active_days_count'
]

X = df[feature_cols].copy()

print("\n--- Feature Vector Summary ---")
print(f"Feature matrix shape: {X.shape}")
print(f"Null count per feature:\n{X.isnull().sum()}")
X.head()

Found 18 partition file(s) for the fact table.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw page-level aggregated data shape: (427292, 8)

--- Feature Vector Summary ---
Feature matrix shape: (427292, 9)
Null count per feature:
clicks_last_30d           0
impressions_last_30d      0
pageviews_last_30d        0
position_last_30d         0
click_decay_ratio         0
impression_decay_ratio    0
ctr_last_30d              0
ctr_gap                   0
active_days_count         0
dtype: int64


,clicks_last_30d,impressions_last_30d,pageviews_last_30d,position_last_30d,click_decay_ratio,impression_decay_ratio,ctr_last_30d,ctr_gap,active_days_count
0,0.0,0.0,0.0,12.384253,0.0,0.000000,0.0,0.000000,476
1,0.0,84.0,6.0,62.299596,0.0,1.312500,0.0,-0.016051,507
2,0.0,304.0,0.0,72.439466,0.0,0.883721,0.0,-0.013805,520
3,0.0,23.0,6.0,61.207937,0.0,1.095238,0.0,-0.016338,519
4,0.0,103.0,1.0,49.492857,0.0,0.609467,0.0,-0.020205,520


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.